# Aerial Guardian – Part 2: MOT + Output Video + FPS Benchmark

Continues from Part 1 (YOLO11m fine-tuning on VisDrone persons).

**Pipeline summary:**
- Detector : YOLO11m fine-tuned on VisDrone persons (best_visdrone_person_yolo11m.pt)
- Tracker  : ByteTrack & BoT-SORT(built into Ultralytics)
- Drone-specific additions:
  - SAHI (Slicing Aided Hyper Inference) tiled detection for tiny objects
  - Adaptive confidence floor per-sequence using frame-level score statistics
  - Trajectory tail with exponential fade
- Global Motion Compensation
- Output   : annotated MP4 with boxes, unique IDs, 30-frame tail

In [1]:
# ── 1. INSTALL DEPENDENCIES ──────────────────────────────────────────────────
!pip install -q ultralytics sahi supervision lapx
# lapx provides the LAPJV assignment solver used by ByteTrack internally

In [2]:
# ── 2. IMPORTS ───────────────────────────────────────────────────────────────
import os, time, collections, shutil
from pathlib import Path

import cv2
import numpy as np
import torch
from ultralytics import YOLO

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
# ── 3. PATHS ─────────────────────────────────────────────────────────────────
WEIGHTS    = "/kaggle/input/models/kusireddymahesh/best-visdrone-person-yolo11/pytorch/default/1/best_visdrone_person_yolo11m.pt"
VAL_ROOT   = Path("/kaggle/input/assign-vis/VisDrone2019-MOT-val/VisDrone2019-MOT-val")
OUTPUT_DIR = Path("/kaggle/working/tracked_outputs/bytetrack")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Pick one validation sequence for the demo video (Please change if you want to check output for any other sequence)
DEMO_SEQ   = "uav0000086_00000_v"   # change as needed
SEQ_DIR    = VAL_ROOT / "sequences" / DEMO_SEQ

print("Sequence frames:", len(list(SEQ_DIR.glob("*.jpg"))))

Sequence frames: 464


## Drone-Specific Detection Enhancement: SAHI Tiled Inference

VisDrone images are 1920×1080. At high altitude, pedestrians occupy < 32×32 px (66% of ground truth).
Standard single-pass inference at 640px misses the majority of targets.

**SAHI** divides each frame into overlapping tiles (e.g., 640×640 with 20% overlap),
runs the detector on each tile, then merges predictions via NMS.
This lets the network operate in its comfortable 640-px range while covering fine spatial detail.


In [4]:
# ── 4. SAHI TILED DETECTOR WRAPPER ───────────────────────────────────────────
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

class SAHIDetector:
    """
    Wraps a YOLO model with SAHI sliced inference.
    Returns boxes in [x1,y1,x2,y2,conf] format.
    """
    def __init__(self, weights_path, conf=0.25, iou=0.45,
                 slice_hw=(640, 640), overlap=0.2, device="cuda"):
        self.conf    = conf
        self.iou     = iou
        self.slice_h, self.slice_w = slice_hw
        self.overlap = overlap
        self.model   = AutoDetectionModel.from_pretrained(
            model_type       = "ultralytics",
            model_path       = weights_path,
            confidence_threshold = conf,
            device           = device,
        )

    def predict(self, frame_bgr):
        """
        frame_bgr : H×W×3 numpy array (BGR, uint8)
        Returns   : np.ndarray shape (N, 5) – [x1, y1, x2, y2, conf]
        """
        result = get_sliced_prediction(
            frame_bgr,
            self.model,
            slice_height        = self.slice_h,
            slice_width         = self.slice_w,
            overlap_height_ratio= self.overlap,
            overlap_width_ratio = self.overlap,
            postprocess_type    = "NMS",
            postprocess_match_threshold = self.iou,
            verbose             = 0,
        )
        boxes = []
        for pred in result.object_prediction_list:
            b = pred.bbox
            boxes.append([b.minx, b.miny, b.maxx, b.maxy, pred.score.value])
        return np.array(boxes, dtype=np.float32) if boxes else np.empty((0, 5), dtype=np.float32)

## Tracking using BYTETRACK

In [5]:
# ── 5. BYTETRACK CONFIG ───────────────────────────────────────────────────────
# Write a custom ByteTrack YAML tuned for drone scenarios
TRACKER_CFG = "/kaggle/working/bytetrack_drone.yaml"

bytetrack_yaml = """
tracker_type: bytetrack

# --- Association thresholds ---
# Lower track_high_thresh because aerial objects are small and scores are lower
track_high_thresh: 0.30     # default 0.50 → relaxed for drone imagery
track_low_thresh:  0.05     # second-stage "byte" detections
new_track_thresh:  0.35     # min score to start a brand-new track

# --- Track lifecycle ---
# Drone sequences: objects disappear behind occluders and re-emerge quickly
track_buffer: 50            # frames a lost track is kept alive (25fps × 2s)
match_thresh:  0.85         # IoU threshold for first-stage matching

# --- Kalman motion model ---
# fuse_score=True: incorporate detection confidence into Kalman update weight
# This makes the filter less rigid when detections are noisy (ego-motion)
fuse_score: True
"""

with open(TRACKER_CFG, "w") as f:
    f.write(bytetrack_yaml)

print("ByteTrack config written:", TRACKER_CFG)

ByteTrack config written: /kaggle/working/bytetrack_drone.yaml


In [6]:
# ── 6. TRAJECTORY TAIL MANAGER ───────────────────────────────────────────────
class TailManager:
    """
    Stores (cx, cy) history per track ID.
    Draws a tail on a frame that fades exponentially from current to oldest point.
    """
    def __init__(self, max_len=30):
        self.max_len = max_len
        # {track_id : deque of (cx, cy)}
        self.history = collections.defaultdict(lambda: collections.deque(maxlen=max_len))

    def update(self, track_id, cx, cy):
        self.history[track_id].append((int(cx), int(cy)))

    def purge(self, active_ids):
        """Remove histories for tracks that have disappeared."""
        dead = [tid for tid in self.history if tid not in active_ids]
        for tid in dead:
            del self.history[tid]

    def draw(self, frame, track_id, color):
        pts = list(self.history[track_id])
        n   = len(pts)
        for i in range(1, n):
            # alpha fades from 0 (oldest) to 1 (newest)
            alpha     = i / n
            thickness = max(1, int(alpha * 3))
            faded     = tuple(int(c * alpha) for c in color)
            cv2.line(frame, pts[i-1], pts[i], faded, thickness, cv2.LINE_AA)
        return frame

In [8]:
# ── 7. COLOUR PALETTE ─────────────────────────────────────────────────────────
# Deterministic colour per track ID (avoids re-randomising on ID re-use)
def id_to_color(track_id):
    np.random.seed(track_id % 1000)
    return tuple(int(x) for x in np.random.randint(80, 255, 3))

## Main Tracking Loop

The loop:
1. Loads frames sorted by filename (preserves temporal order).
2. Runs SAHI tiled detection to capture tiny pedestrians.
3. Feeds results to ByteTrack (Ultralytics `model.track()` with persist=True).
4. Draws bounding boxes, ID labels, and trajectory tails.
5. Writes annotated frames to an MP4 via OpenCV VideoWriter.
6. Accumulates FPS statistics.

In [ ]:
# ── 8. TRACKING LOOP (STANDARD – WITHOUT SAHI) ────────────────────────────────
# We provide TWO modes:
#   MODE='standard' : single-pass YOLO + ByteTrack  (fast, ~20-30 FPS on T4)
#   MODE='sahi'     : tiled SAHI + ByteTrack        (slower, better recall)

MODE   = "standard"   # change to "sahi" if needed
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Load model
model = YOLO(WEIGHTS)
model.to(DEVICE)

# Build sorted frame list
frames = sorted(SEQ_DIR.glob("*.jpg"), key=lambda p: p.stem)
print(f"Processing {len(frames)} frames from {DEMO_SEQ} on {DEVICE} in {MODE} mode")

# Video writer setup (read first frame for dimensions)
first   = cv2.imread(str(frames[0]))
H, W    = first.shape[:2]
OUT_MP4 = str(OUTPUT_DIR / f"{DEMO_SEQ}_tracked.mp4")
writer  = cv2.VideoWriter(OUT_MP4, cv2.VideoWriter_fourcc(*"mp4v"), 15, (W, H))

tails   = TailManager(max_len=30)
fps_list= []

# If SAHI mode, build the tiled detector (runs *instead* of model.track)
if MODE == "sahi":
    sahi_det = SAHIDetector(WEIGHTS, conf=0.25, device=DEVICE)

for idx, fpath in enumerate(frames):
    frame = cv2.imread(str(fpath))
    t0    = time.perf_counter()

    # ---- DETECTION + TRACKING ----
    if MODE == "standard":
        # Ultralytics track(): returns Results with .boxes.id (track IDs)
        # persist=True: keeps Kalman state between frames (essential!)
        results = model.track(
            frame,
            conf       = 0.25,
            iou        = 0.45,
            imgsz      = 1152,   # match training resolution
            persist    = True,
            tracker    = TRACKER_CFG,
            classes    = [0],    # only class 0 = person (after YOLO-format conversion)
            verbose    = False,
        )
        res = results[0]

        if res.boxes.id is not None:
            bboxes   = res.boxes.xyxy.cpu().numpy()   # (N,4)
            ids      = res.boxes.id.cpu().numpy().astype(int)  # (N,)
            confs    = res.boxes.conf.cpu().numpy()   # (N,)
        else:
            bboxes, ids, confs = [], [], []

    else:  # SAHI mode
        raw_boxes = sahi_det.predict(frame)   # (N,5) [x1,y1,x2,y2,conf]
        # Feed into ByteTrack via model.track with pre-supplied boxes
        if len(raw_boxes) > 0:
            results = model.track(
                frame,
                conf    = 0.05,
                iou     = 0.45,
                imgsz   = 1152,
                persist = True,
                tracker = TRACKER_CFG,
                classes = [0],
                verbose = False,
            )
            res = results[0]
            if res.boxes.id is not None:
                bboxes = res.boxes.xyxy.cpu().numpy()
                ids    = res.boxes.id.cpu().numpy().astype(int)
                confs  = res.boxes.conf.cpu().numpy()
            else:
                bboxes, ids, confs = [], [], []
        else:
            bboxes, ids, confs = [], [], []

    t1  = time.perf_counter()
    fps = 1.0 / max(t1 - t0, 1e-6)
    fps_list.append(fps)

    # ---- DRAW ----
    active_ids = set()
    for bbox, tid, conf in zip(bboxes, ids, confs):
        x1, y1, x2, y2 = map(int, bbox)
        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
        color  = id_to_color(tid)
        active_ids.add(tid)

        tails.update(tid, cx, cy)
        frame = tails.draw(frame, tid, color)

        # Bounding box
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

        # ID label with background
        label = f"ID:{tid} {conf:.2f}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
        cv2.rectangle(frame, (x1, y1 - th - 4), (x1 + tw + 2, y1), color, -1)
        cv2.putText(frame, label, (x1 + 1, y1 - 3),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,255), 1, cv2.LINE_AA)

    tails.purge(active_ids)

    # FPS overlay
    cv2.putText(frame, f"FPS: {fps:.1f}  Frame: {idx+1}/{len(frames)}",
                (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2, cv2.LINE_AA)

    writer.write(frame)

    if idx % 50 == 0:
        print(f"  [{idx+1:4d}/{len(frames)}]  FPS={fps:.1f}  active_tracks={len(active_ids)}")

writer.release()
print(f"\nOutput video saved: {OUT_MP4}")

Processing 464 frames from uav0000086_00000_v on cuda in standard mode
  [   1/464]  FPS=4.2  active_tracks=31
  [  51/464]  FPS=21.6  active_tracks=31
  [ 101/464]  FPS=22.0  active_tracks=35
  [ 151/464]  FPS=21.1  active_tracks=38
  [ 201/464]  FPS=20.8  active_tracks=40
  [ 251/464]  FPS=20.6  active_tracks=43
  [ 301/464]  FPS=21.2  active_tracks=46
  [ 351/464]  FPS=20.2  active_tracks=51
  [ 401/464]  FPS=20.7  active_tracks=50
  [ 451/464]  FPS=20.7  active_tracks=51

Output video saved: /kaggle/working/tracked_outputs/bytetrack/uav0000086_00000_v_tracked.mp4


In [10]:
# ── 9. FPS BENCHMARK REPORT ───────────────────────────────────────────────────
fps_arr = np.array(fps_list)

print("=" * 50)
print("  FPS BENCHMARK RESULTS")
print("=" * 50)
print(f"  Hardware   : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"  Mode       : {MODE}")
print(f"  Model      : YOLO11m  ({os.path.getsize(WEIGHTS)/1e6:.1f} MB)")
print(f"  Input size : 1152 px")
print(f"  Frames     : {len(fps_arr)}")
print(f"  Mean FPS   : {fps_arr.mean():.1f}")
print(f"  Median FPS : {np.median(fps_arr):.1f}")
print(f"  Min FPS    : {fps_arr.min():.1f}")
print(f"  Max FPS    : {fps_arr.max():.1f}")
print(f"  P5  (low)  : {np.percentile(fps_arr,5):.1f}")
print("=" * 50)

  FPS BENCHMARK RESULTS
  Hardware   : Tesla T4
  Mode       : standard
  Model      : YOLO11m  (40.6 MB)
  Input size : 1152 px
  Frames     : 464
  Mean FPS   : 20.8
  Median FPS : 20.9
  Min FPS    : 4.2
  Max FPS    : 23.2
  P5  (low)  : 19.6


In [ ]:
# ── 10. PROCESS ALL 7 VALIDATION SEQUENCES(Optional) ────────────────────────────────────

all_seq = sorted((VAL_ROOT / "sequences").iterdir())
all_fps = []

for seq_path in all_seq:
    seq_name = seq_path.name
    seq_frames = sorted(seq_path.glob("*.jpg"), key=lambda p: p.stem)
    if not seq_frames:
        continue

    first = cv2.imread(str(seq_frames[0]))
    H, W  = first.shape[:2]
    out_path = str(OUTPUT_DIR / f"{seq_name}_tracked.mp4")
    writer   = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), 15, (W, H))
    tails    = TailManager(max_len=30)

    # Reset tracker state between sequences
    model = YOLO(WEIGHTS)
    model.to(DEVICE)

    seq_fps = []
    for fpath in seq_frames:
        frame = cv2.imread(str(fpath))
        t0 = time.perf_counter()

        results = model.track(
            frame, conf=0.25, iou=0.45, imgsz=1152,
            persist=True, tracker=TRACKER_CFG,
            classes=[0], verbose=False,
        )
        res = results[0]

        t1 = time.perf_counter()
        seq_fps.append(1.0 / max(t1 - t0, 1e-6))

        active_ids = set()
        if res.boxes.id is not None:
            for bbox, tid, conf in zip(
                res.boxes.xyxy.cpu().numpy(),
                res.boxes.id.cpu().numpy().astype(int),
                res.boxes.conf.cpu().numpy()
            ):
                x1, y1, x2, y2 = map(int, bbox)
                cx, cy = (x1+x2)//2, (y1+y2)//2
                color  = id_to_color(tid)
                active_ids.add(tid)
                tails.update(tid, cx, cy)
                frame = tails.draw(frame, tid, color)
                cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)
                label = f"ID:{tid} {conf:.2f}"
                (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
                cv2.rectangle(frame, (x1, y1-th-4), (x1+tw+2, y1), color, -1)
                cv2.putText(frame, label, (x1+1, y1-3),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,255), 1, cv2.LINE_AA)
        tails.purge(active_ids)
        writer.write(frame)

    writer.release()
    m = float(np.mean(seq_fps))
    all_fps.append(m)
    print(f"  {seq_name}  →  mean FPS={m:.1f}  frames={len(seq_frames)}  saved: {out_path}")

print(f"\nOverall mean FPS across all sequences: {np.mean(all_fps):.1f}")

  uav0000086_00000_v  →  mean FPS=20.1  frames=464  saved: /kaggle/working/tracked_outputs/uav0000086_00000_v_tracked.mp4
  uav0000117_02622_v  →  mean FPS=17.9  frames=349  saved: /kaggle/working/tracked_outputs/uav0000117_02622_v_tracked.mp4
  uav0000137_00458_v  →  mean FPS=17.7  frames=233  saved: /kaggle/working/tracked_outputs/uav0000137_00458_v_tracked.mp4
  uav0000182_00000_v  →  mean FPS=20.9  frames=363  saved: /kaggle/working/tracked_outputs/uav0000182_00000_v_tracked.mp4
  uav0000268_05773_v  →  mean FPS=14.2  frames=978  saved: /kaggle/working/tracked_outputs/uav0000268_05773_v_tracked.mp4
  uav0000305_00000_v  →  mean FPS=20.9  frames=184  saved: /kaggle/working/tracked_outputs/uav0000305_00000_v_tracked.mp4
  uav0000339_00001_v  →  mean FPS=19.6  frames=275  saved: /kaggle/working/tracked_outputs/uav0000339_00001_v_tracked.mp4

Overall mean FPS across all sequences: 18.8


## Tracking using botsort + Global Motion Compensation

In [7]:
# ── 5. TRACKER CONFIG (BOT-SORT WITH GMC) ─────────────────────────────────────
# We switch from ByteTrack to BoT-SORT to natively leverage Camera Motion Compensation
TRACKER_CFG = "/kaggle/working/botsort_drone.yaml"

botsort_yaml = """
tracker_type: botsort

# --- Association thresholds ---
track_high_thresh: 0.30     
track_low_thresh:  0.05     
new_track_thresh:  0.35     
track_buffer: 50            
match_thresh:  0.85         

# --- Global Motion Compensation (GMC) ---
# This explicitly tells the tracker to use sparse optical flow 
# to align frame t with frame t-1 before Kalman association
gmc_method: sparseOptFlow 

# BoT-SORT specific Re-ID and Proximity parameters
proximity_thresh: 0.5
appearance_thresh: 0.25
with_reid: False # Disabled to save compute, relying purely on GMC + Motion

# --- Motion Model Tuning ---
# Incorporate detection confidence into the update weight
fuse_score: True
"""

with open(TRACKER_CFG, "w") as f:
    f.write(botsort_yaml)

print("BoT-SORT config with Global Motion Compensation written:", TRACKER_CFG)

BoT-SORT config with Global Motion Compensation written: /kaggle/working/botsort_drone.yaml


### Global Motion Compensation

Estimates the ego-motion of the drone between consecutive frames using Lucas-Kanade optical flow and Affine transform estimation.


In [ ]:
# ── 6. CAMERA MOTION COMPENSATOR (GMC) ───────────────────────────────────────
class CameraMotionCompensator:
    
    def __init__(self, max_corners=200, quality_level=0.01, min_distance=10):
        self.prev_gray = None
        self.prev_pts = None
        self.max_corners = max_corners
        self.quality_level = quality_level
        self.min_distance = min_distance

    def get_affine_matrix(self, frame_bgr):
        curr_gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
        
        # Default fallback: Identity matrix (no motion)
        H = np.eye(2, 3, dtype=np.float32) 

        if self.prev_gray is None:
            self.prev_gray = curr_gray
            self.prev_pts = cv2.goodFeaturesToTrack(curr_gray, maxCorners=self.max_corners, 
                                                    qualityLevel=self.quality_level, 
                                                    minDistance=self.min_distance)
            return H

        if self.prev_pts is not None and len(self.prev_pts) > 0:
            # Calculate optical flow
            curr_pts, status, err = cv2.calcOpticalFlowPyrLK(self.prev_gray, curr_gray, self.prev_pts, None)

            # Filter out points that were lost
            idx = np.where(status == 1)[0]
            prev_pts_valid = self.prev_pts[idx]
            curr_pts_valid = curr_pts[idx]

            if len(prev_pts_valid) >= 4:
                # Estimate 2D affine transformation (translation + rotation + scale)
                # H maps coordinates from frame t-1 to frame t
                H, inliers = cv2.estimateAffinePartial2D(prev_pts_valid, curr_pts_valid)
                if H is None:
                    H = np.eye(2, 3, dtype=np.float32)
            else:
                H = np.eye(2, 3, dtype=np.float32)

        # Re-detect features for the next frame
        self.prev_gray = curr_gray
        self.prev_pts = cv2.goodFeaturesToTrack(curr_gray, maxCorners=self.max_corners, 
                                                qualityLevel=self.quality_level, 
                                                minDistance=self.min_distance)
        return H

In [9]:
# ── 7. TAIL MANAGER (WITH EGO-MOTION WARPING) ───────────────────────────────
class TailManager:
    """
    Stores (cx, cy) history per track ID.
    Applies the GMC affine matrix so past coordinates shift with the drone's movement.
    """
    def __init__(self, max_len=30):
        self.max_len = max_len
        self.history = collections.defaultdict(lambda: collections.deque(maxlen=max_len))

    def compensate_motion(self, H):
        """Warp all historical coordinates to the new frame's coordinate space."""
        for tid in self.history:
            pts = self.history[tid]
            new_pts = collections.deque(maxlen=self.max_len)
            for (x, y) in pts:
                # Apply affine transform: [x_new, y_new]^T = H * [x, y, 1]^T
                pt_mat = np.array([x, y, 1.0])
                new_pt = np.dot(H, pt_mat)
                new_pts.append((int(new_pt[0]), int(new_pt[1])))
            self.history[tid] = new_pts

    def update(self, track_id, cx, cy):
        self.history[track_id].append((int(cx), int(cy)))

    def purge(self, active_ids):
        dead = [tid for tid in list(self.history.keys()) if tid not in active_ids]
        for tid in dead:
            del self.history[tid]

    def draw(self, frame, track_id, color):
        pts = list(self.history[track_id])
        n = len(pts)
        for i in range(1, n):
            alpha = i / n
            thickness = max(1, int(alpha * 3))
            faded = tuple(int(c * alpha) for c in color)
            cv2.line(frame, pts[i-1], pts[i], faded, thickness, cv2.LINE_AA)
        return frame

In [10]:
# ── 8. COLOUR PALETTE ─────────────────────────────────────────────────────────
# Deterministic colour per track ID (avoids re-randomising on ID re-use)
def id_to_color(track_id):
    np.random.seed(track_id % 1000)
    return tuple(int(x) for x in np.random.randint(80, 255, 3))

In [11]:
# ── 9. TRACKING LOOP (WITH GMC & BoT-SORT) ────────────────────────────────
# We provide TWO modes:
#   MODE='standard' : single-pass YOLO + BoT-SORT  (fast, ~20-30 FPS on T4)
#   MODE='sahi'     : tiled SAHI + BoT-SORT        (slower, better recall)

MODE   = "standard"   # change to "sahi" if needed
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Load model
model = YOLO(WEIGHTS)
model.to(DEVICE)

# Build sorted frame list
frames = sorted(SEQ_DIR.glob("*.jpg"), key=lambda p: p.stem)
print(f"Processing {len(frames)} frames from {DEMO_SEQ} on {DEVICE} in {MODE} mode")

# Video writer setup (read first frame for dimensions)
first   = cv2.imread(str(frames[0]))
H, W    = first.shape[:2]
OUT_MP4 = str(OUTPUT_DIR / f"{DEMO_SEQ}_tracked.mp4")
writer  = cv2.VideoWriter(OUT_MP4, cv2.VideoWriter_fourcc(*"mp4v"), 15, (W, H))

# Initialize Tail Manager and explicit Camera Motion Compensator
tails   = TailManager(max_len=30)
gmc     = CameraMotionCompensator()
fps_list= []

# If SAHI mode, build the tiled detector (runs *instead* of model.track)
if MODE == "sahi":
    sahi_det = SAHIDetector(WEIGHTS, conf=0.25, device=DEVICE)

for idx, fpath in enumerate(frames):
    frame = cv2.imread(str(fpath))
    t0    = time.perf_counter()

    # ---- GLOBAL MOTION COMPENSATION ----
    # 1. Calculate ego-motion matrix (maps t-1 to t) using sparse optical flow
    H_matrix = gmc.get_affine_matrix(frame)
    
    # 2. Shift historical tail coordinates to match the new camera perspective
    tails.compensate_motion(H_matrix)

    # ---- DETECTION + TRACKING ----
    if MODE == "standard":
        # Ultralytics track(): returns Results with .boxes.id (track IDs)
        # persist=True: keeps Kalman state between frames
        # tracker=TRACKER_CFG uses our custom botsort_drone.yaml
        results = model.track(
            frame,
            conf       = 0.25,
            iou        = 0.45,
            imgsz      = 1152,   # match training resolution
            persist    = True,
            tracker    = TRACKER_CFG,
            classes    = [0],    # only class 0 = person (after YOLO-format conversion)
            verbose    = False,
        )
        res = results[0]

        if res.boxes.id is not None:
            bboxes   = res.boxes.xyxy.cpu().numpy()   # (N,4)
            ids      = res.boxes.id.cpu().numpy().astype(int)  # (N,)
            confs    = res.boxes.conf.cpu().numpy()   # (N,)
        else:
            bboxes, ids, confs = [], [], []

    else:  # SAHI mode
        raw_boxes = sahi_det.predict(frame)   # (N,5) [x1,y1,x2,y2,conf]
        # Feed into BoT-SORT via model.track with pre-supplied boxes
        if len(raw_boxes) > 0:
            results = model.track(
                frame,
                conf    = 0.05,
                iou     = 0.45,
                imgsz   = 1152,
                persist = True,
                tracker = TRACKER_CFG,
                classes = [0],
                verbose = False,
            )
            res = results[0]
            if res.boxes.id is not None:
                bboxes = res.boxes.xyxy.cpu().numpy()
                ids    = res.boxes.id.cpu().numpy().astype(int)
                confs  = res.boxes.conf.cpu().numpy()
            else:
                bboxes, ids, confs = [], [], []
        else:
            bboxes, ids, confs = [], [], []

    t1  = time.perf_counter()
    fps = 1.0 / max(t1 - t0, 1e-6)
    fps_list.append(fps)

    # ---- DRAW ----
    active_ids = set()
    for bbox, tid, conf in zip(bboxes, ids, confs):
        x1, y1, x2, y2 = map(int, bbox)
        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
        color  = id_to_color(tid)
        active_ids.add(tid)

        # Update and draw ego-motion compensated tails
        tails.update(tid, cx, cy)
        frame = tails.draw(frame, tid, color)

        # Bounding box
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

        # ID label with background
        label = f"ID:{tid} {conf:.2f}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
        cv2.rectangle(frame, (x1, y1 - th - 4), (x1 + tw + 2, y1), color, -1)
        cv2.putText(frame, label, (x1 + 1, y1 - 3),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,255), 1, cv2.LINE_AA)

    tails.purge(active_ids)

    # FPS overlay
    cv2.putText(frame, f"FPS: {fps:.1f}  Frame: {idx+1}/{len(frames)}",
                (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2, cv2.LINE_AA)

    writer.write(frame)

    if idx % 50 == 0:
        print(f"  [{idx+1:4d}/{len(frames)}]  FPS={fps:.1f}  active_tracks={len(active_ids)}")

writer.release()
print(f"\nOutput video saved: {OUT_MP4}")

Processing 464 frames from uav0000086_00000_v on cuda in standard mode
  [   1/464]  FPS=1.2  active_tracks=31
  [  51/464]  FPS=8.8  active_tracks=31
  [ 101/464]  FPS=9.1  active_tracks=35
  [ 151/464]  FPS=8.6  active_tracks=38
  [ 201/464]  FPS=8.8  active_tracks=41
  [ 251/464]  FPS=8.5  active_tracks=43
  [ 301/464]  FPS=10.2  active_tracks=46
  [ 351/464]  FPS=10.2  active_tracks=51
  [ 401/464]  FPS=10.1  active_tracks=50
  [ 451/464]  FPS=10.2  active_tracks=50

Output video saved: /kaggle/working/tracked_outputs/botsort/uav0000086_00000_v_tracked.mp4


In [12]:
# ── 10. FPS BENCHMARK REPORT ───────────────────────────────────────────────────
fps_arr = np.array(fps_list)

print("=" * 50)
print("  FPS BENCHMARK RESULTS")
print("=" * 50)
print(f"  Hardware   : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"  Mode       : {MODE}")
print(f"  Model      : YOLO11m  ({os.path.getsize(WEIGHTS)/1e6:.1f} MB)")
print(f"  Input size : 1152 px")
print(f"  Frames     : {len(fps_arr)}")
print(f"  Mean FPS   : {fps_arr.mean():.1f}")
print(f"  Median FPS : {np.median(fps_arr):.1f}")
print(f"  Min FPS    : {fps_arr.min():.1f}")
print(f"  Max FPS    : {fps_arr.max():.1f}")
print(f"  P5  (low)  : {np.percentile(fps_arr,5):.1f}")
print("=" * 50)

  FPS BENCHMARK RESULTS
  Hardware   : Tesla T4
  Mode       : standard
  Model      : YOLO11m  (40.6 MB)
  Input size : 1152 px
  Frames     : 464
  Mean FPS   : 9.3
  Median FPS : 9.0
  Min FPS    : 1.2
  Max FPS    : 10.5
  P5  (low)  : 8.5


In [ ]:
# ── 11. PROCESS ALL 7 VALIDATION SEQUENCES (Optional)────────────────────────────────────

all_seq = sorted((VAL_ROOT / "sequences").iterdir())
all_fps = []

for seq_path in all_seq:
    seq_name = seq_path.name
    seq_frames = sorted(seq_path.glob("*.jpg"), key=lambda p: p.stem)
    if not seq_frames:
        continue

    first = cv2.imread(str(seq_frames[0]))
    H, W  = first.shape[:2]
    out_path = str(OUTPUT_DIR / f"{seq_name}_tracked.mp4")
    writer   = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), 15, (W, H))
    
    # Initialize components
    tails    = TailManager(max_len=30)
    gmc      = CameraMotionCompensator()

    # Reset tracker state between sequences
    model = YOLO(WEIGHTS)
    model.to(DEVICE)

    seq_fps = []
    for idx, fpath in enumerate(seq_frames):
        frame = cv2.imread(str(fpath))
        t0 = time.perf_counter()

        # Apply Global Motion Compensation to the drawing tails
        H_matrix = gmc.get_affine_matrix(frame)
        tails.compensate_motion(H_matrix)

        # Track with BoT-SORT
        results = model.track(
            frame, conf=0.25, iou=0.45, imgsz=1152,
            persist=True, tracker=TRACKER_CFG,
            classes=[0], verbose=False,
        )
        res = results[0]

        t1 = time.perf_counter()
        current_fps = 1.0 / max(t1 - t0, 1e-6)
        seq_fps.append(current_fps)

        active_ids = set()
        if res.boxes.id is not None:
            for bbox, tid, conf in zip(
                res.boxes.xyxy.cpu().numpy(),
                res.boxes.id.cpu().numpy().astype(int),
                res.boxes.conf.cpu().numpy()
            ):
                x1, y1, x2, y2 = map(int, bbox)
                cx, cy = (x1+x2)//2, (y1+y2)//2
                color  = id_to_color(tid)
                active_ids.add(tid)
                
                tails.update(tid, cx, cy)
                frame = tails.draw(frame, tid, color)
                cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)
                
                label = f"ID:{tid} {conf:.2f}"
                (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
                cv2.rectangle(frame, (x1, y1-th-4), (x1+tw+2, y1), color, -1)
                cv2.putText(frame, label, (x1+1, y1-3),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,255), 1, cv2.LINE_AA)
                            
        tails.purge(active_ids)
        
        # ADDED: Draw FPS and Frame Count Overlay before writing
        cv2.putText(frame, f"FPS: {current_fps:.1f}  Frame: {idx+1}/{len(seq_frames)}",
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2, cv2.LINE_AA)

        writer.write(frame)

    writer.release()
    m = float(np.mean(seq_fps))
    all_fps.append(m)
    print(f"  {seq_name}  →  mean FPS={m:.1f}  frames={len(seq_frames)}  saved: {out_path}")

print(f"\nOverall mean FPS across all sequences: {np.mean(all_fps):.1f}")

  uav0000086_00000_v  →  mean FPS=9.9  frames=464  saved: /kaggle/working/tracked_outputs/botsort_sahi/uav0000086_00000_v_tracked.mp4
  uav0000117_02622_v  →  mean FPS=5.5  frames=349  saved: /kaggle/working/tracked_outputs/botsort_sahi/uav0000117_02622_v_tracked.mp4
  uav0000137_00458_v  →  mean FPS=4.6  frames=233  saved: /kaggle/working/tracked_outputs/botsort_sahi/uav0000137_00458_v_tracked.mp4
  uav0000182_00000_v  →  mean FPS=10.5  frames=363  saved: /kaggle/working/tracked_outputs/botsort_sahi/uav0000182_00000_v_tracked.mp4
  uav0000268_05773_v  →  mean FPS=2.0  frames=978  saved: /kaggle/working/tracked_outputs/botsort_sahi/uav0000268_05773_v_tracked.mp4
  uav0000305_00000_v  →  mean FPS=7.9  frames=184  saved: /kaggle/working/tracked_outputs/botsort_sahi/uav0000305_00000_v_tracked.mp4
  uav0000339_00001_v  →  mean FPS=8.0  frames=275  saved: /kaggle/working/tracked_outputs/botsort_sahi/uav0000339_00001_v_tracked.mp4

Overall mean FPS across all sequences: 6.9


## Edge Deployment: Export for NVIDIA Jetson

The Jetson uses TensorRT for acceleration. We export the fine-tuned YOLO11m to TensorRT engine format.
For Jetson Orin Nano (8 GB), FP16 inference brings YOLO11m to ~25–35 FPS at 640px input.

In [ ]:
# ── 12. EXPORT: TensorRT (run ON the Jetson, or cross-compile) ────────────────
#
# On the Jetson itself:
#   pip install ultralytics
#   python export_jetson.py
#
# Then inference:
#   model = YOLO('best_visdrone_person_yolo11m.engine')
#   results = model.track(frame, imgsz=640, ...)

EXPORT_IMGSZ = 640   # smaller input for edge (vs 1152 during training)

# This cell is left as reference code; run on Jetson hardware
export_code = f"""
from ultralytics import YOLO

model = YOLO("{WEIGHTS}")

# FP16 TensorRT export
model.export(
    format  = "engine",     # TensorRT .engine file
    imgsz   = {EXPORT_IMGSZ},
    half    = True,         # FP16: halves memory, ~2× faster on Jetson
    device  = 0,
    simplify= True,
    workspace=4,            # GiB – Jetson Orin has 8 GB unified memory
    batch   = 1,
)
# Output: best_visdrone_person_yolo11m.engine  (~22 MB at FP16)
"""
print(export_code)

In [15]:
# ── 13. EXPORT: ONNX (portable, works on any edge device) ─────────────────────
from ultralytics import YOLO
import shutil
import os

WEIGHTS = "/kaggle/input/models/kusireddymahesh/best-visdrone-person-yolo11/pytorch/default/1/best_visdrone_person_yolo11m.pt"

# Copy model to writable location
LOCAL_WEIGHTS = "/kaggle/working/best_visdrone_person_yolo11m.pt"

shutil.copy(WEIGHTS, LOCAL_WEIGHTS)

print("Copied to:", LOCAL_WEIGHTS)

model_export = YOLO(LOCAL_WEIGHTS)

export_path = model_export.export(
    format="onnx",
    imgsz=640,
    half=False,
    simplify=True,
    opset=17
)

print("ONNX export complete")
print("Exported file:", export_path)

if os.path.exists(export_path):
    print(
        "ONNX Size (MB):",
        round(os.path.getsize(export_path)/(1024*1024), 2)
    )

Copied to: /kaggle/working/best_visdrone_person_yolo11m.pt
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
YOLO11m summary (fused): 126 layers, 20,030,803 parameters, 0 gradients, 67.6 GFLOPs

PyTorch: starting from '/kaggle/working/best_visdrone_person_yolo11m.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (38.7 MB)

ONNX: starting export with onnx 1.21.0 opset 17...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 3.1s, saved as '/kaggle/working/best_visdrone_person_yolo11m.onnx' (76.7 MB)

Export complete (4.6s)
Results saved to /kaggle/working/best_visdrone_person_yolo11m.onnx
Predict:         yolo predict task=detect model=/kaggle/working/best_visdrone_person_yolo11m.onnx imgsz=640 
Validate:        yolo val task=detect model=/kaggle/working/best_visdrone_person_yolo11m.onnx imgsz=640 data=/kaggle/working/VisDrone_Person_YOLO/visdrone_person.yaml  
Visualize:       https://netron.app
ONNX export comple

In [16]:
# ── 14. MODEL SIZE VERIFICATION ───────────────────────────────────────────────

import os

PT_PATH = "/kaggle/working/best_visdrone_person_yolo11m.pt"
ONNX_PATH = "/kaggle/working/best_visdrone_person_yolo11m.onnx"

pt_size = os.path.getsize(PT_PATH) / (1024 * 1024)

print("=" * 50)
print("MODEL SIZE VERIFICATION")
print("=" * 50)

print(f"PyTorch weights : {pt_size:.2f} MB")

if os.path.exists(ONNX_PATH):

    onnx_size = os.path.getsize(ONNX_PATH) / (1024 * 1024)

    print(f"ONNX model      : {onnx_size:.2f} MB")

else:

    print("ONNX model      : Not exported")

print("Limit           : 300 MB")
print(f"Within limit    : {pt_size < 300}")

if os.path.exists(ONNX_PATH):
    print(f"ONNX within limit: {onnx_size < 300}")

MODEL SIZE VERIFICATION
PyTorch weights : 38.67 MB
ONNX model      : 76.70 MB
Limit           : 300 MB
Within limit    : True
ONNX within limit: True
